# Real data: two ways to analytically marginalize timing

Notebook **01** introduced three dispositions per fitpar: sample, delta-flat
marginalization, or z-prior marginalization. This notebook focuses on the two
**analytical** choices, on real EPTA DR2 J1640+2224.

If you only know Enterprise/`TimingModel` or Discovery’s improper timing GP, you
have been using the **delta-flat** measure (improper flat prior on the physical
offset). `nltiming` also offers a **z-prior** measure: linearize in the
prior-whitened coordinate and integrate a proper unit-normal. Same named axes,
different evidence — they must not be treated as interchangeable.

**Setup:** MetaPulsar is required today for the `TimingPulsar` contract (see
notebook **01**). J1640 par/tim are expected under a MetaPulsar checkout at
`data/ipta-dr2/.../J1640+2224`.


In [ ]:
import os
os.environ.setdefault("JAX_ENABLE_X64", "1")
from pathlib import Path


import discovery as ds
from discovery import transport as dst
from metapulsar import create_metapulsar
from metapulsar.sandbox_tempo2 import configure_logging
from nltiming import (
    NonLinearTimingModel, TimingInference, WhiteningConfig,
)

ds.config(kernels="metamath")
configure_logging(level="WARNING")

# Real IPTA-DR2 J1640+2224 (EPTA v2.2). Find the repo root from the CWD.
_here = Path.cwd()
_repo = next(p for p in (_here, *_here.parents) if (p / "data" / "ipta-dr2").is_dir())
PAR = _repo / "data/ipta-dr2/EPTA_v2.2/J1640+2224/J1640+2224.par"
TIM = _repo / "data/ipta-dr2/EPTA_v2.2/J1640+2224/J1640+2224_all.tim"
assert PAR.exists(), f"J1640 par not found: {PAR}"

mp = create_metapulsar(
    {"epta": [{"par": str(PAR), "tim": str(TIM), "timing_package": "tempo2"}]},
    use_pulse_numbers="no",
)
nd = {f"{mp.name}_efac": 1.0, f"{mp.name}_log10_t2equad": -8.0}
reference = dst.reference_noise_frozen(
    ds.makenoise_measurement_simple(mp, nd), nd, description="J1640 WN reference")
center = {f"{mp.name}_rednoise_log10_A": -14.0, f"{mp.name}_rednoise_gamma": 3.5}
print(f"{mp.name}: {len(mp.toas)} TOAs, {len(mp.fitpars)} fitpars")
print("fitpars:", list(mp.fitpars))


## 1. Same DM axes, two marginalization measures

Marginalize the dispersion family once as delta-flat and once as z-prior.
Everything else is sampled. The resolved plans — and their fingerprints —
differ because the likelihood normalization differs.


In [ ]:
subset = [p for p in ("DM", "DM1", "DM2") if p in mp.fitpars]

ctx_df = NonLinearTimingModel(
    engines="jug", inference=TimingInference.groups(delta_flat=subset), name="timing"
).for_pulsar(mp)
ctx_zp = NonLinearTimingModel(
    engines="jug", inference=TimingInference.groups(z_prior=subset), name="timing"
).for_pulsar(mp)

print("subset:", subset)
print("delta-flat: marg_delta =", ctx_df.plan.marginalized_delta,
      "| marg_z =", ctx_df.plan.marginalized_z)
print("z-prior   : marg_delta =", ctx_zp.plan.marginalized_delta,
      "| marg_z =", ctx_zp.plan.marginalized_z)
print("\ndistinct plan fingerprints:",
      ctx_df.plan.fingerprint() != ctx_zp.plan.fingerprint())
print("  delta-flat:", ctx_df.plan.fingerprint()[:32])
print("  z-prior   :", ctx_zp.plan.fingerprint()[:32])


## 2. Different measure ⇒ different log-likelihood

At the same physical point (timing delays at the par-file reference, red noise
integrated at a fixed hyper), the two models disagree on the marginalized
log-likelihood. Internally: delta-flat uses an improper GP; z-prior uses a
proper unit-normal GP. That is a scientific choice, not an implementation detail.


In [ ]:
nd_hyper = {**nd, f"{mp.name}_rednoise_log10_A": -14.0, f"{mp.name}_rednoise_gamma": 3.5}
for label, ctx_m in (("delta-flat", ctx_df), ("z-prior", ctx_zp)):
    psl = ds.PulsarLikelihood([
        mp.residuals, ds.makenoise_measurement_simple(mp, nd),
        ds.makegp_fourier(mp, ds.powerlaw, 10, name="rednoise"),
        *ctx_m.discovery_signals(joint=False)])
    # timing delay keys at the reference (delta=0); marginalized blocks integrated.
    params = dict(nd_hyper)
    for k in ctx_m.delay_keys:
        params[k] = 0.0
    print(f"{label:10s} logL(reference) = {float(psl.logL(params)):.6g}")
print("\n-> the two marginalizations are distinct models: the improper flat-in-δ")
print("   measure and the proper unit-normal measure give different evidence.")


## 3. Optionally sample the z-prior coefficients (Enterprise)

By default a z-prior block is analytically integrated. Pass
`sample_z_coefficients=True` on `enterprise_signal` to promote those
coefficients to sampled unit-normal parameters instead — useful when you want
them in a joint latent vector. Here the DM-family coefficients show up in
`pta.param_names`.


In [ ]:
from enterprise.signals import parameter, signal_base, white_signals

ntm_zp = NonLinearTimingModel(
    engines="jug", inference=TimingInference.groups(z_prior=subset),
    whitening=WhiteningConfig(), name="timing")
white = white_signals.MeasurementNoise(efac=parameter.Constant(1.0))
pta = signal_base.PTA([(white + ntm_zp.enterprise_signal(sample_z_coefficients=True))(mp)])
coeff = [p for p in pta.param_names if "zprior_coefficients" in p]
print("z-prior coefficient parameters:", coeff)
print("all timing parameters:", [p for p in pta.param_names if "timing" in p])


## Summary

- Delta-flat and z-prior marginalization are different models (improper vs proper
  measure). Pick deliberately; fingerprints keep run products honest.
- Z-prior coefficients can stay integrated or be sampled
  (`sample_z_coefficients=True`).
- Disposition (sample / delta-flat / z-prior) is independent of identical
  linearity — geometry of *sampled* axes is the topic of notebooks **02**/**03**.
